# Notebook 3: Generate SCORE Critique-Correction Data

**Input:** GSM8K train set  
**Output:** `score_valid.jsonl` — critique-correction pairs for Notebook 4  
**Model:** `google/gemma-3-4b-it`

**Before running:** GPU P100 + Internet + HF_TOKEN in Secrets

In [2]:
# # CELL 1: Install
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")

Done.


In [3]:
# CELL 2: Login
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
print("Login successful.")




Login successful.


In [4]:
# CELL 3: Imports + GPU
import os, json, re, random, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

OUTPUT_DIR = "/kaggle/working/score_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch : 2.9.0+cu126
GPU     : Tesla T4
VRAM    : 15.6 GB


In [13]:
# CELL 4: Config
CONFIG = {
    "model_name"       : "google/gemma-3-4b-it",
    "num_questions"    : 400,     # test run — change to 2000 for full run
    "n_solutions"      : 3,
    "random_seed"      : 123,
    "sol_max_tokens"   : 1000,
    "sol_temperature"  : 0.8,
    "crit_max_tokens"  : 350,
    "crit_temperature" : 0.2,
    "raw_file"         : f"{OUTPUT_DIR}/score_raw.jsonl",
    "valid_file"       : f"{OUTPUT_DIR}/score_valid.jsonl",
    "checkpoint_file"  : f"{OUTPUT_DIR}/checkpoint.json",
    "report_file"      : f"{OUTPUT_DIR}/score_report.json",
    "save_every"       : 20,
}
print("Config loaded.")
for k, v in CONFIG.items():
    print(f"  {k:22s}: {v}")

Config loaded.
  model_name            : google/gemma-3-4b-it
  num_questions         : 300
  n_solutions           : 3
  random_seed           : 123
  sol_max_tokens        : 1000
  sol_temperature       : 0.8
  crit_max_tokens       : 350
  crit_temperature      : 0.2
  raw_file              : /kaggle/working/score_data/score_raw.jsonl
  valid_file            : /kaggle/working/score_data/score_valid.jsonl
  checkpoint_file       : /kaggle/working/score_data/checkpoint.json
  report_file           : /kaggle/working/score_data/score_report.json
  save_every            : 20


In [6]:
# CELL 5: Load GSM8K
print("Loading GSM8K...")
gsm8k      = load_dataset("gsm8k", "main")
train_data = gsm8k["train"]

random.seed(CONFIG["random_seed"])
indices      = random.sample(range(len(train_data)), CONFIG["num_questions"])
sampled_data = [train_data[i] for i in indices]

print(f"GSM8K train : {len(train_data)}")
print(f"Sampled     : {len(sampled_data)}")
print(f"\nExample: {sampled_data[0]['question'][:100]}")

Loading GSM8K...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

GSM8K train : 7473
Sampled     : 200

Example: A frog lays her eggs over a series of 4 days.  The first day she lays 50 eggs.  The second day, she 


In [7]:
# CELL 6: Load Gemma 3 4B
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
import torch

print(f"Loading {CONFIG['model_name']}...")

model = Gemma3ForConditionalGeneration.from_pretrained(
    CONFIG["model_name"],
    device_map="auto"
).eval()

processor = AutoProcessor.from_pretrained(CONFIG["model_name"])


Loading google/gemma-3-4b-it...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [8]:
# CELL 7: Generate Function + Answer Extractors
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
import torch

print(f"Loading {CONFIG['model_name']}...")

model = Gemma3ForConditionalGeneration.from_pretrained(
    CONFIG["model_name"],
    device_map="auto"
).eval()

processor = AutoProcessor.from_pretrained(CONFIG["model_name"])

def generate_single(messages, max_tokens, temperature):
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            do_sample          = True,
            temperature        = temperature,
            top_p              = 0.95,
            repetition_penalty = 1.1,
        )

    new_tokens = output[0][input_len:]
    return processor.decode(new_tokens, skip_special_tokens=True).strip()


def extract_gt_answer(answer_str):
    m = re.search(r"####\s*(-?[\d,]+)", answer_str)
    return m.group(1).replace(",", "") if m else ""

def extract_pred_answer(text):
    m = re.search(r"####\s*(-?[\d,]+)", text)
    if m: return m.group(1).replace(",", "")
    m = re.search(r"(?:the answer is|answer:|result:)\s*\$?(-?[\d,]+)", text, re.IGNORECASE)
    if m: return m.group(1).replace(",", "")
    numbers = re.findall(r"-?[\d,]+", text)
    return numbers[-1].replace(",", "") if numbers else ""


# Quick sanity check
print(f"\nModel loaded! GPU used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Device : {model.device}")

Loading google/gemma-3-4b-it...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]


Model loaded! GPU used: 4.26 GB
Device : cuda:0


In [9]:
# CELL 8: Prompt Templates

SOLUTION_SYSTEM = """You are a math problem solver.
Solve the problem step by step.
End your answer with: #### [number]"""

CRITIQUE_SYSTEM = """You are a math error checker.
You are given a wrong solution and the correct answer.
Find the FIRST step that contains an error.

Reply EXACTLY in this format:
Error step: [copy the wrong step exactly]
Reason: [one sentence why it is wrong]
Correction: [corrected version of that step]"""

CORRECTION_SYSTEM = """You are a math problem solver.
A previous attempt contained an error described below.
Solve the problem correctly from scratch, step by step.
End with: #### [number]"""


def solution_prompt(q):
    return [
        {"role": "system", "content": [{"type": "text", "text": SOLUTION_SYSTEM}]},
        {"role": "user",   "content": [{"type": "text", "text": f"Problem: {q}"}]},
    ]

def critique_prompt(q, wrong_sol, correct_ans):
    return [
        {"role": "system", "content": [{"type": "text", "text": CRITIQUE_SYSTEM}]},
        {"role": "user",   "content": [{"type": "text", "text": (
            f"Problem: {q}\n\n"
            f"Wrong solution:\n{wrong_sol}\n\n"
            f"Correct answer: {correct_ans}\n\n"
            f"Find the first error:"
        )}]},
    ]

def correction_prompt(q, critique):
    return [
        {"role": "system", "content": [{"type": "text", "text": CORRECTION_SYSTEM}]},
        {"role": "user",   "content": [{"type": "text", "text": (
            f"Problem: {q}\n\n"
            f"Error found in previous attempt:\n{critique}\n\n"
            f"Solve correctly:"
        )}]},
    ]

print("Prompts ready.")

Prompts ready.


In [10]:
import time

start = time.time()

# Test one generation
print("Testing single solution generation...")
test_item = sampled_data[0]
test_q    = test_item["question"]
test_gt   = extract_gt_answer(test_item["answer"])
test_sol  = generate_single(solution_prompt(test_q), CONFIG["sol_max_tokens"], CONFIG["sol_temperature"])
test_pred = extract_pred_answer(test_sol)


print(f"total Time : {(time.time() - start):.2f} minutes")


Testing single solution generation...
total Time : 34.16 minutes


In [12]:
print(f"total Time : {((time.time() - start)/60):.2f} minutes")

total Time : 1.21 minutes


In [11]:
print(f"Q         : {test_q}...")
print(f"GT answer : {test_gt}")
print(f"Predicted : {test_pred}")
print(f"Correct   : {test_gt == test_pred}")
print(f"\nSolution:\n{test_sol}")

Q         : A frog lays her eggs over a series of 4 days.  The first day she lays 50 eggs.  The second day, she doubles her production of eggs.  The third day she lays 20 more than the second day, and the last day she doubles the first three days total.  How many eggs did the frog lay over the span of the 4 days?...
GT answer : 810
Predicted : 810
Correct   : True

Solution:
Let $E_1$ be the number of eggs laid on the first day, $E_2$ be the number of eggs laid on the second day, $E_3$ be the number of eggs laid on the third day, and $E_4$ be the number of eggs laid on the fourth day. We are given that $E_1 = 50$.

On the second day, the frog doubles her production of eggs from the first day, so $E_2 = 2 \times E_1 = 2 \times 50 = 100$.

On the third day, the frog lays 20 more eggs than the second day, so $E_3 = E_2 + 20 = 100 + 20 = 120$.

On the fourth day, the frog doubles the total number of eggs laid in the first three days. The total number of eggs laid in the first three days is

In [14]:
# CELL 9: Validation Functions

def is_valid_critique(text):
    if not text or len(text.strip()) < 20:
        return False, "too_short"
    if not re.search(r"error step:", text, re.IGNORECASE):
        return False, "missing_error_step"
    if not re.search(r"reason:", text, re.IGNORECASE):
        return False, "missing_reason"
    if not re.search(r"correction:", text, re.IGNORECASE):
        return False, "missing_correction"
    m = re.search(r"reason:\s*(.+)", text, re.IGNORECASE)
    if m and len(m.group(1).split()) < 4:
        return False, "reason_too_vague"
    return True, "ok"

def is_valid_correction(text, gt_answer):
    if not text or len(text.strip()) < 20:
        return False, "too_short"
    pred = extract_pred_answer(text)
    if not pred:
        return False, "no_answer"
    if pred != gt_answer:
        return False, f"wrong_ans_{pred}"
    return True, "ok"

# Tests
good = "Error step: 5 x 4 = 25\nReason: Multiplying 5 by 4 gives 20 not 25.\nCorrection: 5 x 4 = 20"
bad  = "That seems wrong."
print(f"Good → {is_valid_critique(good)}")
print(f"Bad  → {is_valid_critique(bad)}")

Good → (True, 'ok')
Bad  → (False, 'too_short')


In [15]:
# CELL 10: Main Generation Loop
# Runtime: ~3-4 hours for 500 questions on P100
# Checkpoints every 20 questions

from tqdm.notebook import tqdm

print(f"Starting: {len(sampled_data)} questions x {CONFIG['n_solutions']} solutions each")
print("-" * 55)

all_results = []
start_idx   = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["raw_file"]):
        with open(CONFIG["raw_file"]) as f:
            all_results = [json.loads(l) for l in f if l.strip()]
    print(f"Resumed: idx={start_idx}, saved={len(all_results)}")
else:
    print("Starting fresh...")

for q_idx in tqdm(range(start_idx, len(sampled_data)), desc="Questions"):
    item      = sampled_data[q_idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    if not gt_answer:
        continue

    # Step 1: generate N solutions
    solutions = []
    for _ in range(CONFIG["n_solutions"]):
        try:
            sol  = generate_single(solution_prompt(question), CONFIG["sol_max_tokens"], CONFIG["sol_temperature"])
            pred = extract_pred_answer(sol)
            solutions.append({"text": sol, "pred": pred, "correct": pred == gt_answer})
        except Exception:
            break

    wrong_solutions = [s for s in solutions if not s["correct"] and s["pred"]]

    if not wrong_solutions:
        all_results.append({
            "question": question, "gt_answer": gt_answer,
            "n_solutions": len(solutions), "n_wrong": 0,
            "pairs": [], "status": "all_correct"
        })
        continue

    # Step 2: critique + correction (max 2 per question)
    pairs = []
    for ws in wrong_solutions[:2]:
        try:
            critique = generate_single(
                critique_prompt(question, ws["text"], gt_answer),
                CONFIG["crit_max_tokens"], CONFIG["crit_temperature"]
            )
            cv, cr = is_valid_critique(critique)
            if not cv:
                pairs.append({"final_valid": False, "status": f"crit_{cr}"})
                continue

            corrected = generate_single(
                correction_prompt(question, critique),
                CONFIG["sol_max_tokens"], CONFIG["crit_temperature"]
            )
            ov, orr = is_valid_correction(corrected, gt_answer)

            pairs.append({
                "wrong_solution" : ws["text"],
                "wrong_answer"   : ws["pred"],
                "critique"       : critique,
                "corrected"      : corrected,
                "final_valid"    : cv and ov,
                "status"         : orr if not ov else "ok",
            })
        except Exception:
            break

    all_results.append({
        "question"    : question,
        "gt_answer"   : gt_answer,
        "n_solutions" : len(solutions),
        "n_wrong"     : len(wrong_solutions),
        "pairs"       : pairs,
        "status"      : "processed",
    })

    if (q_idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["raw_file"], "w") as f:
            for r in all_results: f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": q_idx + 1}, f)       

with open(CONFIG["raw_file"], "w") as f:
    for r in all_results: f.write(json.dumps(r) + "\n")

print(f"\nDone. Processed: {len(all_results)} questions.")

Starting: 200 questions x 3 solutions each
-------------------------------------------------------
Starting fresh...


Questions:   0%|          | 0/200 [00:00<?, ?it/s]


Done. Processed: 200 questions.


In [16]:
# CELL 11: Extract Valid Pairs + Save

valid_pairs = []
for result in all_results:
    if result["status"] != "processed":
        continue
    for pair in result["pairs"]:
        if pair.get("final_valid"):
            valid_pairs.append({
                "question"       : result["question"],
                "gt_answer"      : result["gt_answer"],
                "wrong_solution" : pair["wrong_solution"],
                "critique"       : pair["critique"],
                "corrected"      : pair["corrected"],
            })

with open(CONFIG["valid_file"], "w") as f:
    for pair in valid_pairs:
        f.write(json.dumps(pair) + "\n")

print(f"Valid pairs : {len(valid_pairs)}")
print(f"Saved       : {CONFIG['valid_file']}")
print("\n" + "="*60)
print("SAMPLE PAIRS")
print("="*60)
for i, pair in enumerate(valid_pairs[:2]):
    print(f"\n--- Example {i+1} ---")
    print(f"Q         : {pair['question'][:90]}...")
    print(f"GT        : {pair['gt_answer']}")
    print(f"Wrong sol : {pair['wrong_solution'][:100]}...")
    print(f"Critique  :\n{pair['critique']}")
    print(f"Corrected : ...{pair['corrected'][-80:]}")

Valid pairs : 24
Saved       : /kaggle/working/score_data/score_valid.jsonl

SAMPLE PAIRS

--- Example 1 ---
Q         : Ariel began fencing in 2006. If she was born in 1992 and has been fencing for 16 years, ho...
GT        : 30
Wrong sol : Here's how to solve the problem step-by-step:

1. **Find her age when she started fencing:** She was...
Critique  :
Error step: She started fencing in 2006 and has been fencing for 16 years. So, her current age is 2006 + 16 = 2022 years old.
Reason: Adding 16 years to her starting year (2006) incorrectly calculates her current age because it doesn’t account for her birth year.
Correction: She started fencing in 2006 and has been fencing for 16 years. Therefore, her current age is 2006 + 16 = 2022 - 1992 = 30 years old.
Corrected : ... between the current year and her birth year:
Age $= 2022 - 1992 = 30$.

#### 30

--- Example 2 ---
Q         : Andrew installed hardwood flooring in his house. His bedroom took eight wooden planks, his...
GT        : 

In [17]:
# CELL 12: Final Report

total_q     = len(all_results)
all_correct = sum(1 for r in all_results if r["status"] == "all_correct")
processed   = sum(1 for r in all_results if r["status"] == "processed")
total_wrong = sum(r.get("n_wrong", 0) for r in all_results)

fail_reasons = {}
for r in all_results:
    for p in r.get("pairs", []):
        if not p.get("final_valid"):
            s = p.get("status", "unknown")
            fail_reasons[s] = fail_reasons.get(s, 0) + 1

survival = len(valid_pairs) / max(1, total_wrong) * 100

print("=" * 60)
print("SCORE DATA GENERATION REPORT")
print("=" * 60)
print(f"Questions processed   : {total_q}")
print(f"All correct (skipped) : {all_correct}")
print(f"Had wrong solutions   : {processed}")
print(f"Total wrong solutions : {total_wrong}")
print(f"Valid pairs saved     : {len(valid_pairs)}")
print(f"Survival rate         : {survival:.1f}%  (paper baseline: ~25%)")

if fail_reasons:
    print("\nFailure breakdown:")
    for r, c in sorted(fail_reasons.items(), key=lambda x: -x[1]):
        print(f"  {r:35s}: {c}")

report = {
    "model"          : CONFIG["model_name"],
    "questions"      : total_q,
    "valid_pairs"    : len(valid_pairs),
    "survival_rate"  : round(survival, 2),
    "failure_reasons": fail_reasons,
}
with open(CONFIG["report_file"], "w") as f:
    json.dump(report, f, indent=2)

print(f"\nReport: {CONFIG['report_file']}")
print("\n" + "=" * 60)
if len(valid_pairs) >= 300:
    print(f"GOOD: {len(valid_pairs)} pairs — proceed to Notebook 4")
elif len(valid_pairs) >= 100:
    print(f"MARGINAL: {len(valid_pairs)} — scale num_questions to 2000")
else:
    print(f"LOW: {len(valid_pairs)} — share report for debugging")
print("=" * 60)

SCORE DATA GENERATION REPORT
Questions processed   : 200
All correct (skipped) : 160
Had wrong solutions   : 40
Total wrong solutions : 74
Valid pairs saved     : 24
Survival rate         : 32.4%  (paper baseline: ~25%)

Failure breakdown:
  wrong_ans_6                        : 4
  no_answer                          : 3
  wrong_ans_400                      : 2
  wrong_ans_650                      : 2
  wrong_ans_200                      : 2
  wrong_ans_252                      : 2
  wrong_ans_0                        : 2
  wrong_ans_31                       : 1
  wrong_ans_57                       : 1
  wrong_ans_60                       : 1
  wrong_ans_54                       : 1
  crit_missing_correction            : 1
  wrong_ans_62                       : 1
  wrong_ans_83                       : 1
  wrong_ans_3                        : 1
  wrong_ans_250                      : 1
  wrong_ans_22                       : 1
  wrong_ans_17                       : 1
  wrong_ans_12        